# Monitor del `allTransmissionCodes.json` (2da vuelta) — notebook

Ejecuta de forma interactiva lo que hace `monitor_codes_json.py`, reutilizando sus
funciones (no duplica lógica), y añade vistas: **curva de crecimiento** y **matriz de
transiciones de estado**.

> Coloca `monitor_codes_json.py` en el mismo directorio que este notebook
> (o ajusta la ruta en la celda de configuración).

In [2]:
import sys, json, gzip, time, importlib
from pathlib import Path
from collections import Counter

# Si el .py está en otra carpeta, añade aquí su ruta:
# sys.path.append(r"C:\\Users\\Julian\\SIGMODEL\\scan_e14_colombia\\2vlta")
import monitor_codes_json as M
importlib.reload(M)

OUT = Path("codes_snapshots")          # carpeta de snapshots (la misma del script)
URL = M.URL                            # URL del JSON de 2da vuelta
UA  = "Mozilla/5.0 (compatible; E14CodesMonitor/1.0)"
REF = "https://e14segundavueltapresidente.registraduria.gov.co/"
OUT.mkdir(parents=True, exist_ok=True)
print("Snapshots en:", OUT.resolve())

Snapshots en: C:\Users\Julian\SIGMODEL\scan_e14_colombia\2vlta\codes_snapshots


## 1. Una sola descarga (equivale a `--once`)
Descarga el JSON, lo encadena, guarda snapshot y reporta el diff frente a la última vez.

In [ ]:
estado = M.cargar_estado(OUT)
print(f"Estado previo: {estado['n']:,} actas")
raw = M.fetch(URL, (15, 180), 4, 2.0, UA, REF)
estado = M.procesar(OUT, raw, estado, solo_cambios=False)

## 2. Bucle de monitoreo (N descargas)
Para vigilancia 24/7 desatendida es mejor el `.py` con el Programador de tareas; este bucle
es para sesiones interactivas. Interrumpe el kernel para pararlo.

In [ ]:
INTERVALO   = 300     # segundos entre descargas
N_DESCARGAS = 12      # nº de ciclos

estado = M.cargar_estado(OUT)
try:
    for i in range(N_DESCARGAS):
        try:
            raw = M.fetch(URL, (15, 180), 4, 2.0, UA, REF)
            estado = M.procesar(OUT, raw, estado, solo_cambios=False)
        except Exception as e:
            print("ERROR de descarga:", e)
        if i < N_DESCARGAS - 1:
            time.sleep(INTERVALO)
except KeyboardInterrupt:
    print("Detenido por el usuario.")

## 3. Verificar la cadena de integridad
Recorre los eslabones desde GENESIS y comprueba que ningún snapshot ni línea del log fue alterado.

In [3]:
M.verificar(OUT)

Cadena íntegra: 125 eslabones verificados desde GENESIS, sin alteraciones.
  2026-06-22 00:53:34  ->  2026-06-22 11:18:07
  actas: 78,928  ->  121,629


## 4. Comparar dos snapshots
Acepta nombre de fichero, `'latest'`, o un trozo de fecha. Reporta añadidas / eliminadas /
hash cambiado / estado cambiado y vuelca el detalle a un CSV.

In [4]:
snaps = sorted(OUT.glob("allTransmissionCodes_*.json.gz"))
print(f"{len(snaps)} snapshots")
if snaps:
    print("primero:", snaps[0].name, "| último:", snaps[-1].name)
    M.comparar(OUT, snaps[0].name, "latest")

125 snapshots
primero: allTransmissionCodes_20260622_005334.json.gz | último: allTransmissionCodes_20260622_111807.json.gz
A = allTransmissionCodes_20260622_005334.json.gz  (78,928 actas)
B = latest.json  (121,629 actas)

  añadidas en B:    42,701
  ELIMINADAS en B:  0
  hash cambiado:    0
  estado cambiado:  1,261

Detalle volcado en: comparacion_allTransmissionCodes_20260622_005334__latest.csv


## 5. Curva de crecimiento
Lee `_monitor_log.csv` y dibuja el nº de actas en el tiempo + los incrementos por descarga.

In [5]:
import pandas as pd, matplotlib.pyplot as plt

log = pd.read_csv(OUT / "_monitor_log.csv", parse_dates=["timestamp"])
log = log[log["n_actas"] > 0].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(log["timestamp"], log["n_actas"], lw=1.6)
ax.set_title("Actas cargadas en el tiempo"); ax.set_xlabel("hora"); ax.set_ylabel("nº actas")
ax.grid(alpha=.3); fig.autofmt_xdate(); plt.show()

cols = ["timestamp", "n_actas", "anadidas", "eliminadas", "hash_cambiado", "estado_cambiado", "alerta"]
display(log[cols].tail(15))

alertas = log[log["alerta"] == 1]
print(f"\nDescargas con ALERTA: {len(alertas)}")
if len(alertas):
    display(alertas[cols])


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## 6. Estados: distribución y matriz de transiciones
Responde a "¿qué significa *N cambian estado*?": qué estados existen ahora y qué transiciones
(de qué estado a qué estado) ocurrieron entre dos momentos.

In [6]:
# Distribución de estados en el último snapshot
idx_now = M.indexar(json.loads((OUT / "latest.json").read_text(encoding="utf-8")))
dist = Counter(st for _, st in idx_now.values())
print("Distribución de estados (latest):")
for st, c in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  estado {st!r}: {c:,}")

Distribución de estados (latest):
  estado '11': 121,546
  estado '3': 83


In [7]:
import pandas as pd

def matriz_transiciones(out, ref_a, ref_b):
    a = M.indexar(json.loads(M._leer_snapshot(M._resolver(out, ref_a))))
    b = M.indexar(json.loads(M._leer_snapshot(M._resolver(out, ref_b))))
    pares = [(a[k][1], b[k][1]) for k in (set(a) & set(b)) if a[k][1] != b[k][1]]
    if not pares:
        print("Sin cambios de estado entre A y B."); return None
    return pd.crosstab(pd.Series([p[0] for p in pares], name="antes"),
                       pd.Series([p[1] for p in pares], name="ahora"))

snaps = sorted(OUT.glob("allTransmissionCodes_*.json.gz"))
if len(snaps) >= 2:
    m = matriz_transiciones(OUT, snaps[0].name, "latest")
    print("Transiciones de estado (filas=antes, columnas=ahora):")
    display(m)
    print("\nLas transiciones hacia adelante son rutina; vigila retrocesos o cambios tras publicación.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Julian\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import